# A10 side study: return vs critic and audience reviews (Tanner Edwards)

**Personal side notebook** (2026-11-26). Same data as `Edwards-A10.ipynb` (`data/clean/films_clean.csv`), same rows: `in_population` films from the **training years 2000–2015**, so the 2016–19 test films stay unseen.

**Why a separate notebook:** reviews exist only **after** release, so they can never be model inputs (contract A6). This is a **second outcome** to study, not part of the money-loser model.

**Tanner's hypotheses**
1. Films critics love don't usually make much money.
2. Critics and audiences often disagree, and the disagreement itself is interesting: what does its "bell curve" look like, from *critics loved it, audiences didn't* to *audiences loved it, critics didn't*?

**Scores used:** Rotten Tomatoes' **Tomatometer** (% of critics who rated it positive) and **Audience Score** (% of audience ratings that were positive), both from the RT movie datasets (andrezaza first, then stefanoleone992). Same site and same kind of scale, so they can be compared directly. *Not* the pipeline's `audience_score`, which mostly comes from TMDB's vote average (a 1–10 average on a much narrower scale). **Return** = US box office ÷ budget, as in the main notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 50)
BLUE, BLUE_LIGHT, ORANGE, INK2 = "#2a78d6", "#cde2fb", "#eb6834", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.titlesize": 10, "axes.titlelocation": "left", "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8})

films = pd.read_csv(Path("../data/clean/films_clean.csv"), low_memory=False)
tr = films[films["in_population"] & films["year"].le(2015)].copy()
tr["critic"] = tr["rt_tomatoMeter"].fillna(tr["sl_tomatometer_rating"])
tr["audience"] = tr["rt_audienceScore"].fillna(tr["sl_audience_rating"])
tr["gap"] = tr["audience"] - tr["critic"]          # + = audiences liked it more than critics
tr["winner"] = ~tr["loser"].astype(bool)
r = tr.dropna(subset=["critic", "audience"]).copy()
print(f"Training films: {len(tr):,}; with both RT scores: {len(r):,} ({len(r) / len(tr):.0%})")
print(f"Spearman correlation, critic vs audience: {r['critic'].corr(r['audience'], method='spearman'):.2f}")
print(f"Spearman correlation with return: critic {r['critic'].corr(r['us_multiple'], method='spearman'):.2f}, "
      f"audience {r['audience'].corr(r['us_multiple'], method='spearman'):.2f}, gap {r['gap'].corr(r['us_multiple'], method='spearman'):.2f}")

## 1. How critics and audiences score films

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
bins = np.arange(0, 105, 5)
ax[0].hist(r["critic"], bins=bins, color=BLUE, alpha=.8, label="critics (Tomatometer)")
ax[0].hist(r["audience"], bins=bins, histtype="step", lw=2, color=ORANGE, label="audiences (Audience Score)")
ax[0].set(title="Score distributions", xlabel="% positive", ylabel="films"); ax[0].legend(fontsize=8)
ax[1].scatter(r["critic"], r["audience"], s=5, alpha=.25, color=BLUE)
ax[1].plot([0, 100], [0, 100], color="#9a9890", ls="--", lw=1)
ax[1].set(title="Critic vs audience (dashed = they agree)", xlabel="critics % positive", ylabel="audiences % positive", xlim=(0, 100), ylim=(0, 100))
ax[2].hist(r["gap"], bins=np.arange(-80, 85, 5), color=BLUE)
ax[2].axvline(0, color=INK2, lw=1)
ax[2].set(title="Disagreement: audience − critic (the 'bell curve')", xlabel="◀ critics liked it more        audiences liked it more ▶", ylabel="films")
plt.tight_layout(); plt.show()
q = r["gap"].quantile([.05, .25, .5, .75, .95]).round(0)
print(f"Gap (audience − critic): median {q[.5]:+.0f}, middle half {q[.25]:+.0f} to {q[.75]:+.0f}, 5th–95th percentile {q[.05]:+.0f} to {q[.95]:+.0f}")
print(f"Audiences more positive than critics: {(r['gap'] > 0).mean():.0%} of films; strongly (+25 or more): {(r['gap'] >= 25).mean():.0%}; "
      f"critics strongly more positive (−25 or less): {(r['gap'] <= -25).mean():.0%}")

## 2. Return vs critic score and vs audience score

Median return per 10-point score band (dot) with the middle half of films (bar). Dashed line = 1.25× winner line.

In [ ]:
def band_plot(ax, score, title):
    b = pd.cut(r[score], range(0, 101, 10), include_lowest=True, labels=[f"{i}-{i + 10}" for i in range(0, 100, 10)])
    t = r.groupby(b, observed=True)["us_multiple"].agg(films="size", med="median", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75))
    x = np.arange(len(t))
    ax.vlines(x, t["q25"], t["q75"], color=BLUE_LIGHT, lw=7); ax.plot(x, t["med"], "o-", color=BLUE, ms=5)
    ax.axhline(1.25, color="#9a9890", ls="--", lw=1)
    ax.set_yscale("log"); ax.set_yticks([0.05, 0.1, 0.25, 0.5, 1.25, 2, 4], ["0.05x", "0.1x", "0.25x", "0.5x", "1.25x", "2x", "4x"]); ax.minorticks_off()
    ax.set_xticks(x, [f"{i}\nn={n}" for i, n in zip(t.index, t["films"])], fontsize=7)
    ax.set(title=title, xlabel="% positive", ylabel="US box office / budget")
    return t

fig, ax = plt.subplots(1, 2, figsize=(15, 4.3), sharey=True)
tc = band_plot(ax[0], "critic", "Return by CRITIC score")
ta = band_plot(ax[1], "audience", "Return by AUDIENCE score")
plt.tight_layout(); plt.show()
display(pd.DataFrame({"critic: median return": tc["med"].round(2), "critic: films": tc["films"],
                      "audience: median return": ta["med"].round(2), "audience: films": ta["films"]}))

## 3. Tanner's hypothesis: do critics' favorites make money?

Four groups, cut at 60% positive (Rotten Tomatoes' own "fresh" line for critics):
- **Both loved it** (critics ≥ 60, audiences ≥ 60)
- **Critic darlings** (critics ≥ 60, audiences < 60)
- **Crowd pleasers** (critics < 60, audiences ≥ 60)
- **Both disliked it**

Compared on return, winners, and budget (bigger films reach wider audiences, so budget is shown too).

In [ ]:
r["quadrant"] = np.select([(r["critic"] >= 60) & (r["audience"] >= 60), (r["critic"] >= 60), (r["audience"] >= 60)],
                          ["Both loved it", "Critic darlings", "Crowd pleasers"], "Both disliked it")
ORDER = ["Both loved it", "Crowd pleasers", "Critic darlings", "Both disliked it"]
quad = r.groupby("quadrant").agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
                                 winners=("winner", "mean"), median_budget_M=("fin_budget", lambda s: s.median() / 1e6),
                                 median_us_gross_M=("domestic_gross", lambda s: s.median() / 1e6)).reindex(ORDER)
quad["% of films"] = (100 * quad["films"] / len(r)).round(0)
quad["winners"] = (100 * quad["winners"]).round(0).astype(int).astype(str) + "%"
display(quad.round(2))
tier = pd.cut(r["fin_budget"], [0, 10e6, 40e6, np.inf], labels=["under $10M", "$10-40M", "$40M+"])
print("Median return by group, inside budget tiers (same-size films compared):")
display(r.pivot_table(index="quadrant", columns=tier, values="us_multiple", aggfunc="median", observed=True).reindex(ORDER).round(2))
print("Number of films in each cell:")
display(pd.crosstab(r["quadrant"], tier).reindex(ORDER))

## 4. The two tails of the disagreement curve

In [ ]:
r["gap_band"] = pd.cut(r["gap"], [-101, -25, -10, 10, 25, 101],
                       labels=["critics far more positive (≤ −25)", "critics more (−25 to −10)", "agree (±10)",
                               "audiences more (+10 to +25)", "audiences far more positive (≥ +25)"])
gb = r.groupby("gap_band", observed=True).agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
                                             winners=("winner", lambda s: f"{s.mean():.0%}"), median_critic=("critic", "median"),
                                             median_audience=("audience", "median"), median_budget_M=("fin_budget", lambda s: s.median() / 1e6))
display(gb.round(2))
cols = ["title", "year", "critic", "audience", "gap", "fin_budget", "domestic_gross", "us_multiple"]
print("Critics loved it, audiences didn't (most negative gaps, 50+ critic reviews):")
display(r[r["rtrev_n_reviews"].fillna(0) >= 50].nsmallest(10, "gap")[cols])
print("Audiences loved it, critics didn't (most positive gaps, 50+ critic reviews):")
display(r[r["rtrev_n_reviews"].fillna(0) >= 50].nlargest(10, "gap")[cols])

## 5. Which genres drive the disagreement?

In [ ]:
g = r.assign(genre=r["genres_all"].fillna("").str.split(",")).explode("genre")
g["genre"] = g["genre"].str.strip()
gg = g[g["genre"].ne("")].groupby("genre").agg(films=("film_uid", "size"), median_critic=("critic", "median"),
                                                median_audience=("audience", "median"), median_gap=("gap", "median"),
                                                median_return=("us_multiple", "median"))
display(gg[gg["films"] >= 40].sort_values("median_gap").round(2))

## 6. Findings (run 2026-11-26, 3,497 training films with both scores)

**How they score:** critics use the whole 0–100 range almost evenly; audiences bunch in the middle (few films below 30% or above 90%). Critics and audiences mostly agree (Spearman 0.70).

**Hypothesis 1, "critics' favorites don't make much money": supported, with a twist.**
- Critic score barely tracks return (Spearman 0.11). Films from 0% to 80% fresh all sit around 0.56–0.82×; only the top band (90–100%) clearly does better (1.42×).
- Audience score tracks return about twice as strongly (0.24), climbing steadily from 0.20× (10–20%) to 1.30–1.34× (80%+).
- **Critic darlings** (critics ≥ 60, audiences < 60) are the worst-earning group with any acclaim: median 0.48×, 25% winners, median US gross only $7M. Same budget, same story: inside every budget tier they trail films both groups liked.
- **The twist:** acclaim helps when audiences agree. Films **both** groups liked are the best group (1.02×, 43% winners), and **crowd pleasers** critics disliked are close behind (0.94×, 40%). What audiences think matters more for money than what critics think.

**Hypothesis 2, the disagreement curve: it isn't symmetric.** The median film gets **5 points more** from audiences than from critics; the middle half runs from −7 to +20. The audience side has the long tail: 19% of films are liked far more by audiences (+25 or more) vs only 3% liked far more by critics (−25 or less).
- **Audiences far more positive:** median return 0.93× (*Madea* films 9–11×, *Step Up* 5.4×, *Grandma's Boy*).
- **Critics far more positive:** 0.42× (*Mr. Turner*, *Rampart*, *Big Game*). Exceptions are family films critics loved and adult reviewers didn't (*Spy Kids* 3.2×).
- **By genre,** audiences are most generous to Action, Fantasy, Romance, Horror and Comedy (+7 to +10 points); Documentary is the only genre where critics are more positive.

**Caveats**
- Both scores come after release, so they describe outcomes; they can't predict one. Audience scores come from people who chose to see the film (and are probably partly lifted by a film's success), so the link to return runs both ways.
- Possible **second labels** for later: `crowd_pleaser` (audience ≥ 60) or the `gap` itself. A pre-release model could ask "will audiences like it?" using the same inputs as the money model.
- Pipeline note: `audience_score` in `01_clean_dataset.ipynb` takes TMDB's vote average first; Rotten Tomatoes' Audience Score (used here, 96% filled) would be more comparable to the critic score.
